# Senate Stage 2: a small, controlled fundraising signal (experiment)

Does pre-election fundraising add anything beyond fundamentals, polls and expert ratings, especially in competitive
and unpolled races? Production doesn't change: Stage 1 stays `SENATE_S1` and the blend keeps `stage2.WEIGHTS`. The
code is in `src/senate/stage2_fundraising.py` (experimental, not imported by production), tested in
`tests/test_senate_stage2_fundraising.py`.

`04_fundraising_in_blend.ipynb` tested fundraising *inside Stage 1* with the old look-ahead FEC totals. This notebook
adds it *after* the blend instead, with pre-election totals only.

**Fundraising advantage (D-positive):** `fund_adv = log((D receipts + $100k) / (R receipts + $100k))`. Each nominee's
receipts come from the last report before Election Day, normally the pre-general 12G (through about 20 days before).
An amount whose report ends on or after Election Day (the full-cycle summary through December 31) is never used, and
the race then has no fundraising signal. Races without a D or R nominee, independent-vs-R races (Orman, McMullin,
Osborn) and nominees without an FEC record get no signal either. Their money is never treated as $0 or as a Democrat's.

**Variants** (all scored on the same races; every fit uses earlier years only):

| | |
|---|---|
| **Baseline** | the production S1 blend: 0.37 / 0.54 / 0.09 with polls, 0.65 / 0 / 0.35 without |
| **A: fourth component** | 0.35 / 0.54 / 0.09 / **0.02** with polls, 0.60 / 0 / 0.35 / **0.05** without. The fundraising component is a margin, `a + b × fund_adv`, from an OLS fit of actual D margins on `fund_adv` over 2006 through Y−2 |
| **B: capped adjustment** | `M_baseline + ΔF`, with `ΔF = clip(slope × fund_adv, ±cap)`. The cap is **0.5** points with polls and **1.0** without, fixed in advance. The slope is the OLS slope of the production blend's out-of-sample residuals on `fund_adv` over 2012 through Y−2. The intercept (the blend's average bias) is discarded, so ΔF = 0 at even money |

A race without fundraising keeps its baseline prediction under both A and B.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from senate import stage1, stage2
from senate import stage2_fundraising as sf
from senate.simulation import calibration as mc

pd.set_option("display.width", 200)
KEY = sr.KEY

## Settings

In [2]:
STAGE1_CONFIG = stage1.SENATE_S1  # production fundamentals
EVAL_YEARS = [2016, 2018, 2020, 2022, 2024]  # the years with expert ratings, as in 02_blend
STABILIZER = 100_000.0  # dollars added to each side of the log ratio; fixed in advance
CAPS = {"polling": 0.5, "no_polling": 1.0}  # approach B, margin points; fixed in advance, never tuned
WEIGHTS_4 = {  # approach A
    "polling": {"fundamentals": 0.35, "polling": 0.54, "experts": 0.09, "fundraising": 0.02},
    "no_polling": {"fundamentals": 0.60, "polling": 0.00, "experts": 0.35, "fundraising": 0.05},
}
MC_TEST_YEARS = (2018, 2020, 2022, 2024)  # Monte Carlo errors are estimated from 2016 on, as in 03_monte_carlo
print("production weights:", stage2.WEIGHTS)

data = sr.load_data()
polls, ratings = stage2.load_inputs()
fund = sf.fundraising_table(stabilizer=STABILIZER)

production weights: {'polling': {'fundamentals': 0.37, 'polling': 0.54, 'experts': 0.09}, 'no_polling': {'fundamentals': 0.65, 'polling': 0.0, 'experts': 0.35}}


## 1. Pre-election fundraising: coverage and cutoff

Every race in `senate_fec.csv`, by fundraising status. `post_election_only` means a nominee's only FEC amount is
dated after Election Day, almost always in 2006–2016.

In [3]:
scored = data.loc[data["observed"], KEY].merge(fund, on=KEY, how="left", validate="one_to_one")
display(Markdown("**All races**"))
display(pd.crosstab(fund["year"], fund["fund_status"], margins=True))
display(Markdown("**Scored races (observed D-vs-R contests)**"))
pd.crosstab(scored["year"], scored["fund_status"], margins=True)

**All races**

fund_status,independent_race,no_nominee,ok,post_election_only,All
year,,,,,
2006,0,1,8,24,33
2008,0,1,25,9,35
2010,0,1,25,11,37
2012,0,0,27,6,33
2014,1,1,27,7,36
2016,0,1,26,7,34
2018,0,1,33,1,35
2020,0,1,34,0,35
2022,1,0,33,1,35


**Scored races (observed D-vs-R contests)**

fund_status,ok,post_election_only,All
year,,,
2006,7,24,31
2008,25,9,34
2010,24,10,34
2012,27,6,33
2014,27,7,34
2016,26,6,32
2018,33,1,34
2020,34,0,34
2022,32,1,33


**Cutoff check:** the latest report end date used in each year must fall before Election Day. A nominee who filed
nothing at all ($0, no dates) has no end date.

In [4]:
used = fund[fund["fund_status"] == "ok"]
cutoff = used.groupby("year").agg(latest_dem=("dem_report_end", "max"), latest_gop=("gop_report_end", "max"))
cutoff["election_day"] = pd.to_datetime(cutoff.index.map(sf.election_day))
cutoff["days_before"] = (cutoff["election_day"] - cutoff[["latest_dem", "latest_gop"]].max(axis=1)).dt.days
assert (cutoff["days_before"] > 0).all()
cutoff

,latest_dem,latest_gop,election_day,days_before
year,,,,
2006,2006-10-18,2006-10-18,2006-11-07,20
2008,2008-10-18,2008-10-15,2008-11-04,17
2010,2010-10-13,2010-10-13,2010-11-02,20
2012,2012-10-17,2012-10-18,2012-11-06,19
2014,2014-10-17,2014-10-15,2014-11-04,18
2016,2016-10-19,2016-10-27,2016-11-08,12
2018,2018-10-17,2018-10-17,2018-11-06,20
2020,2020-10-14,2020-10-14,2020-11-03,20
2022,2022-10-19,2022-10-19,2022-11-08,20


In [5]:
display(Markdown("**fund_adv on the scored races** (log points; +1 is about 2.7× the opponent's receipts)"))
scored.groupby("year")["fund_adv"].describe().round(2)

**fund_adv on the scored races** (log points; +1 is about 2.7× the opponent's receipts)

,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
2006,7.0,0.75,1.53,-0.45,-0.12,0.01,1.18,3.59
2008,25.0,0.12,2.24,-3.66,-1.16,-0.43,1.57,4.43
2010,24.0,-0.21,1.56,-3.40,-0.60,-0.12,0.93,2.48
2012,27.0,-0.01,2.04,-4.60,-0.63,0.26,1.23,3.47
2014,27.0,-0.08,1.93,-2.96,-1.66,-0.04,0.90,4.20
2016,26.0,0.06,2.21,-4.07,-0.89,-0.05,1.20,4.17
2018,33.0,0.95,1.54,-1.94,0.17,0.92,1.83,3.96
2020,34.0,0.48,1.61,-3.69,-0.28,0.67,1.13,3.37
2022,32.0,-0.07,2.50,-5.50,-2.61,0.48,1.47,4.22


## 2. Calibration, chronological

Each eval year Y uses fits from earlier years only:
- **B_slope:** blend residual per log point. Its window starts in 2012, the first year with out-of-sample Stage 1
  predictions. 2012–2014 have no ratings, so their blend is the production fallback (fundamentals and polls only).
- **S1_slope:** the same fit on the Stage 1 residuals, for the fundamentals-level diagnostic.
- **A:** the margin-on-fundraising regression, from 2006.

In [6]:
rows, calibration_table = sf.backtest(EVAL_YEARS, data=data, polls=polls, ratings=ratings, fund=fund,
                                      config=STAGE1_CONFIG, caps=CAPS, weights=WEIGHTS_4)
rows["prediction_B_uncapped"] = rows["prediction"] + rows["delta_F_uncapped"].fillna(0)  # diagnostic only
calibration_table.round(2)

,B_slope,B_slope_se,B_n,B_years,S1_slope,S1_slope_se,A_intercept,A_slope,A_n,A_years
year,,,,,,,,,,
2016,1.99,0.57,54,2012-2014,3.47,0.75,0.36,11.56,110,2006-2014
2018,1.81,0.43,80,2012-2016,3.07,0.57,-0.80,11.23,136,2006-2016
2020,1.42,0.34,113,2012-2018,2.57,0.49,-1.27,11.09,169,2006-2018
2022,1.31,0.30,147,2012-2020,2.41,0.42,-3.00,10.99,203,2006-2020
2024,1.15,0.25,179,2012-2022,1.94,0.34,-3.19,10.47,235,2006-2022


**Baseline unchanged:**
- the baseline here equals the production out-of-sample blend (`mc.blend_out_of_sample`, which 03_monte_carlo uses);
- races without fundraising keep it under A and B;
- B never moves a race by more than its cap.

In [7]:
production = mc.blend_out_of_sample(STAGE1_CONFIG, years=EVAL_YEARS, data=data, polls=polls, ratings=ratings)
check = rows.merge(production[[*KEY, "prediction"]], on=KEY, how="outer", suffixes=("", "_production"),
                   validate="one_to_one", indicator=True)
assert (check["_merge"] == "both").all()
assert np.allclose(check["prediction"], check["prediction_production"], atol=1e-12, rtol=0)
no_fund = rows["fund_adv"].isna()
assert (rows.loc[no_fund, "prediction_A"] == rows.loc[no_fund, "prediction"]).all()
assert (rows.loc[no_fund, "prediction_B"] == rows.loc[no_fund, "prediction"]).all()
cap = np.where(rows["polling_available"], CAPS["polling"], CAPS["no_polling"])
assert (rows["delta_F"].abs() <= cap + 1e-12).all()
print(f"{len(rows)} races scored by every variant; {int((~no_fund).sum())} with pre-election fundraising, "
      f"{int(no_fund.sum())} without ({rows.loc[no_fund, 'fund_status'].value_counts().to_dict()})")

166 races scored by every variant; 158 with pre-election fundraising, 8 without ({'post_election_only': 8})


## 3. Results, pooled 2016–2024

- `bias` = mean(actual − prediction): negative means too Democratic.
- `ex_ante`: |partisan_lean + eve generic ballot| ≤ 10, known before the election.
- `B uncapped` is a diagnostic, not a candidate: it shows what the caps hold back.
- **Fundamentals level:** S1 alone and S1 + the uncapped fundraising slope (fitted on Stage 1 residuals). This shows
  whether a gain exists only in the regression.

In [8]:
PREDICTIONS = {"baseline (production)": "prediction", "A: 4th component": "prediction_A",
               "B: capped adjustment": "prediction_B"}
DIAGNOSTICS = {"B uncapped (diagnostic)": "prediction_B_uncapped", "S1 fundamentals only": "margin_fundamentals",
               "S1 + fundraising (uncapped)": "fundamentals_adj"}
stage2.score_table(rows, data, {**PREDICTIONS, **DIAGNOSTICS}).round(2)

,n,mae,bias,ex_ante_n,ex_ante_mae,accuracy_pct,ex_ante_accuracy_pct,bias_inc_R,bias_inc_open,bias_inc_D
baseline (production),166.0,4.76,-1.86,51.0,3.58,93.37,84.31,-3.42,-1.66,-0.55
A: 4th component,166.0,4.71,-1.98,51.0,3.54,93.98,84.31,-3.56,-1.74,-0.66
B: capped adjustment,166.0,4.77,-2.00,51.0,3.61,94.58,86.27,-3.15,-1.67,-1.12
B uncapped (diagnostic),166.0,5.09,-2.48,51.0,3.48,94.58,86.27,-2.00,-1.64,-3.31
S1 fundamentals only,166.0,6.01,-1.11,51.0,4.98,92.17,80.39,-3.92,-1.34,1.52
S1 + fundraising (uncapped),166.0,6.49,-2.18,51.0,4.55,93.98,86.27,-1.39,-1.33,-3.29


### Paired comparison with the baseline

Per race: |error| of the variant minus |error| of the baseline (negative is better). The SE treats the five election
years as the independent units, since errors within a year are correlated.

In [9]:
def paired(rows, column, subset=None):
    r = rows if subset is None else rows[subset]
    diff = (r["actual"] - r[column]).abs() - r["residual"].abs()
    yearly = diff.groupby(r["year"]).mean()
    return pd.Series({"n": len(r), "moved": int((r[column] != r["prediction"]).sum()),
                      "better": int((diff < -1e-9).sum()), "worse": int((diff > 1e-9).sum()),
                      "mean_diff": diff.mean(), "se_by_year": yearly.std(ddof=1) / np.sqrt(len(yearly)),
                      "years_better": int((yearly < 0).sum()), "years": len(yearly)})

ex_ante = rows.merge(data[[*KEY, "partisan_lean", "gb_margin_2p_norm"]], on=KEY, how="left", validate="one_to_one")
ex_ante = ((ex_ante["partisan_lean"] + ex_ante["gb_margin_2p_norm"]).abs() <= sr.EX_ANTE_THRESHOLD).to_numpy()
SUBSETS = {"all": None, "ex-ante competitive": ex_ante, "polled": rows["polling_available"].to_numpy(),
           "unpolled": ~rows["polling_available"].to_numpy()}
pd.DataFrame({(label, name): paired(rows, column, mask) for label, column in list(PREDICTIONS.items())[1:]
              for name, mask in SUBSETS.items()}).T.round(3)

n  moved  better  worse  mean_diff  se_by_year  years_better  years
A: 4th component     all                  166.0  158.0    81.0   77.0     -0.052       0.053           3.0    5.0
                     ex-ante competitive   51.0   48.0    26.0   22.0     -0.048       0.031           4.0    5.0
                     polled               109.0  103.0    52.0   51.0     -0.034       0.050           4.0    5.0
                     unpolled              57.0   55.0    29.0   26.0     -0.086       0.108           3.0    5.0
B: capped adjustment all                  166.0  158.0    75.0   83.0      0.011       0.084           2.0    5.0
                     ex-ante competitive   51.0   48.0    22.0   26.0      0.021       0.117           2.0    5.0
                     polled               109.0  103.0    45.0   58.0      0.050       0.109           3.0    5.0
                     unpolled              57.0   55.0    30.0   25.0     -0.064       0.142           3.0    5.0

## 4. Polled and unpolled races

In [10]:
SHOW = ["n", "mae", "bias", "ex_ante_n", "ex_ante_mae", "accuracy_pct", "ex_ante_accuracy_pct"]
sf.scores_by(rows.assign(polled=np.where(rows["polling_available"], "polled", "unpolled")), data,
             {**PREDICTIONS, "S1 fundamentals only": "margin_fundamentals",
              "S1 + fundraising (uncapped)": "fundamentals_adj"}, "polled")[SHOW].round(2)

n   mae  bias  ex_ante_n  ex_ante_mae  accuracy_pct  ex_ante_accuracy_pct
polled   variant                                                                                                   
polled   baseline (production)        109.0  4.35 -1.92       48.0         3.65         89.91                 83.33
         A: 4th component             109.0  4.32 -2.00       48.0         3.61         90.83                 83.33
         B: capped adjustment         109.0  4.40 -2.17       48.0         3.72         91.74                 85.42
         S1 fundamentals only         109.0  6.12 -0.87       48.0         4.94         88.07                 79.17
         S1 + fundraising (uncapped)  109.0  6.34 -2.40       48.0         4.61         90.83                 85.42
unpolled baseline (production)         57.0  5.54 -1.75        3.0         2.52        100.00                100.00
         A: 4th component              57.0  5.46 -1.94        3.0         2.36        100.00                100.00
         B: capped adjustment          57.0  5.48 -1.67        3.0         1.84        100.00                100.00
         S1 fundamentals only          57.0  5.79 -1.58        3.0         5.65        100.00                100.00
         S1 + fundraising (uncapped)   57.0  6.80 -1.76        3.0         3.53        100.00                100.00

## 5. By election year

In [11]:
by_year = sf.scores_by(rows, data, {**PREDICTIONS, "S1 fundamentals only": "margin_fundamentals",
                                    "S1 + fundraising (uncapped)": "fundamentals_adj"}, "year")
display(Markdown("**MAE, all races**"))
display(by_year["mae"].unstack("year").round(2))
display(Markdown("**MAE, ex-ante competitive** (n per year: "
                 + ", ".join(f"{y} {int(n)}" for y, n in by_year.xs("baseline (production)", level="variant")
                             ["ex_ante_n"].items()) + ")"))
display(by_year["ex_ante_mae"].unstack("year").round(2))
display(Markdown("**Bias (actual − prediction)**"))
display(by_year["bias"].unstack("year").round(2))
display(Markdown("**Winners called, %**"))
by_year["accuracy_pct"].unstack("year").round(1)

**MAE, all races**

year,2016,2018,2020,2022,2024
variant,,,,,
baseline (production),7.05,3.38,5.13,4.28,4.06
A: 4th component,6.93,3.28,5.24,4.10,4.09
B: capped adjustment,6.86,3.43,5.29,4.10,4.27
S1 fundamentals only,9.13,6.39,4.98,5.17,4.49
S1 + fundraising (uncapped),8.80,7.32,5.18,5.87,5.38


**MAE, ex-ante competitive** (n per year: 2016 12, 2018 9, 2020 10, 2022 9, 2024 11)

year,2016,2018,2020,2022,2024
variant,,,,,
baseline (production),5.05,3.01,2.93,3.94,2.76
A: 4th component,4.99,2.94,2.99,3.80,2.73
B: capped adjustment,5.02,3.03,3.19,3.53,2.98
S1 fundamentals only,8.27,4.66,3.13,4.67,3.60
S1 + fundraising (uncapped),8.01,3.71,3.51,3.46,3.29


**Bias (actual − prediction)**

year,2016,2018,2020,2022,2024
variant,,,,,
baseline (production),-4.06,0.44,-4.34,0.88,-2.27
A: 4th component,-4.09,0.34,-4.58,0.81,-2.41
B: capped adjustment,-4.01,0.15,-4.53,0.86,-2.51
S1 fundamentals only,-5.14,1.74,-2.73,1.41,-1.00
S1 + fundraising (uncapped),-5.31,-1.08,-3.98,1.57,-2.17


**Winners called, %**

year,2016,2018,2020,2022,2024
variant,,,,,
baseline (production),93.8,91.2,94.1,90.9,97.0
A: 4th component,93.8,94.1,94.1,90.9,97.0
B: capped adjustment,93.8,94.1,94.1,93.9,97.0
S1 fundamentals only,87.5,91.2,97.1,90.9,93.9
S1 + fundraising (uncapped),93.8,91.2,94.1,97.0,93.9


## 6. Adjustment distributions

How far each variant moves the baseline, in margin points (D-positive), by polling case. `at_cap` is the share of
races with fundraising whose uncapped adjustment reaches the cap.

In [12]:
moves = rows.assign(case=np.where(rows["polling_available"], "polled", "unpolled"),
                    shift_A=rows["prediction_A"] - rows["prediction"])
has_fund = moves["fund_adv"].notna()
QUANTILES = [0.05, 0.25, 0.5, 0.75, 0.95]
dist = {}
for case, g in moves[has_fund].groupby("case"):
    for label, col in {"ΔF (B)": "delta_F", "ΔF uncapped": "delta_F_uncapped", "A − baseline": "shift_A"}.items():
        s = g[col]
        dist[(case, label)] = {"n": len(s), "mean": s.mean(), "mean_abs": s.abs().mean(),
                               **{f"q{int(q * 100)}": s.quantile(q) for q in QUANTILES}, "min": s.min(),
                               "max": s.max()}
pd.DataFrame(dist).T.round(2)

n  mean  mean_abs    q5   q25   q50   q75   q95   min   max
polled   ΔF (B)        103.0  0.27      0.46 -0.50  0.16  0.50  0.50  0.50 -0.50  0.50
         ΔF uncapped   103.0  0.93      2.00 -3.41  0.16  1.07  1.71  5.42 -6.90  8.33
         A − baseline  103.0  0.09      0.23 -0.42 -0.09  0.07  0.25  0.59 -0.77  0.84
unpolled ΔF (B)         55.0 -0.08      0.96 -1.00 -1.00 -0.71  1.00  1.00 -1.00  1.00
         ΔF uncapped    55.0  0.13      3.31 -5.63 -2.80 -0.71  3.72  5.64 -8.13  6.61
         A − baseline   55.0  0.20      0.61 -1.14 -0.23  0.22  0.79  1.17 -2.27  1.46

In [13]:
display(Markdown("**Share of races at the cap (%), races with fundraising**"))
at_cap = moves[has_fund].groupby(["case", "year"])["at_cap"].mean().unstack("year") * 100
at_cap["pooled"] = moves[has_fund].groupby("case")["at_cap"].mean() * 100
display(at_cap.round(0))
display(Markdown("**Mean ΔF by incumbent party** (Democratic incumbents tend to outraise, so B leans toward "
                 "incumbents)"))
inc = moves.merge(data[[*KEY, "inc_signed"]], on=KEY, how="left", validate="one_to_one")
inc["incumbent"] = inc["inc_signed"].map({-1: "R", 0: "open", 1: "D"})
inc[has_fund.to_numpy()].groupby(["case", "incumbent"])[["delta_F", "shift_A", "residual"]].agg(
    ["count", "mean"]).round(2)

**Share of races at the cap (%), races with fundraising**

year,2016,2018,2020,2022,2024,pooled
case,,,,,,
polled,78.0,88.0,90.0,82.0,82.0,84.0
unpolled,88.0,100.0,93.0,100.0,81.0,91.0


**Mean ΔF by incumbent party** (Democratic incumbents tend to outraise, so B leans toward incumbents)

delta_F       shift_A       residual      
                     count  mean   count  mean    count  mean
case     incumbent                                           
polled   D              46  0.41      46  0.10       46 -1.21
         R              35  0.06      35  0.09       35 -3.91
         open           22  0.30      22  0.06       22 -1.37
unpolled D              21  1.00      21  0.16       21  0.34
         R              23 -0.83      23  0.25       23 -1.20
         open           11 -0.58      11  0.15       11 -2.23

**Races where a variant changes the called winner** (2016–2024):

In [14]:
flips = rows[(np.sign(rows["prediction_A"]) != np.sign(rows["prediction"]))
             | (np.sign(rows["prediction_B"]) != np.sign(rows["prediction"]))]
flips[[*KEY, "polling_available", "fund_adv", "actual", "prediction", "prediction_A", "prediction_B",
       "delta_F"]].round(2)

,year,state,class,polling_available,fund_adv,actual,prediction,prediction_A,prediction_B,delta_F
47,2018,MT,1,True,1.51,3.66,-0.46,0.10,0.04,0.5
121,2022,NV,3,True,1.11,0.80,-0.21,-0.06,0.29,0.5


## 7. Does it survive the Monte Carlo?

The production error model (S-M0) on each variant's out-of-sample predictions, as in `03_monte_carlo`. Each test
year's error SDs come from that variant's own residuals of earlier years (2016 on). Lower Brier is better;
`coverage_0.9` should be near 0.9.

In [15]:
mc_results = {}
for label, column in PREDICTIONS.items():
    variant = production.drop(columns=["prediction", "residual", "center"]).merge(
        rows[[*KEY, column]].rename(columns={column: "prediction"}), on=KEY, validate="one_to_one")
    variant = variant.assign(residual=variant["actual"] - variant["prediction"], center=variant["prediction"])
    mc_results[label] = mc.backtest(variant, model="S-M0", test_years=MC_TEST_YEARS, n_sims=20_000, seed=0)
pd.concat({label: r["metrics"][["n", "brier", "n_competitive", "brier_competitive", "coverage_0.8", "coverage_0.9",
                                "width_0.9"]] for label, r in mc_results.items()}, names=["variant"]).astype(float).round(4)

n   brier  n_competitive  brier_competitive  coverage_0.8  coverage_0.9  width_0.9
variant                                                                                                             
baseline (production) 2018     34.0  0.0602            9.0             0.0923        1.0000        1.0000    32.4411
                      2020     34.0  0.0544           10.0             0.1118        0.8235        0.9412    25.2157
                      2022     33.0  0.0386            9.0             0.1386        0.9091        0.9394    23.7875
                      2024     33.0  0.0312           11.0             0.0852        0.8788        1.0000    22.6286
                      pooled  134.0  0.0463           39.0             0.1060        0.9030        0.9701    26.0602
A: 4th component      2018     34.0  0.0583            9.0             0.0901        1.0000        1.0000    32.2367
                      2020     34.0  0.0553           10.0             0.1139        0.7941        0.9412    24.9576
                      2022     33.0  0.0371            9.0             0.1328        0.9091        0.9394    23.6216
                      2024     33.0  0.0312           11.0             0.0838        0.8788        1.0000    22.4327
                      pooled  134.0  0.0457           39.0             0.1043        0.8955        0.9701    25.8537
B: capped adjustment  2018     34.0  0.0568            9.0             0.0870        1.0000        1.0000    31.9642
                      2020     34.0  0.0568           10.0             0.1168        0.8235        0.9118    25.0004
                      2022     33.0  0.0349            9.0             0.1249        0.8788        0.9394    23.6015
                      2024     33.0  0.0310           11.0             0.0827        0.9091        0.9697    22.4022
                      pooled  134.0  0.0451           39.0             0.1022        0.9030        0.9552    25.7830

## 8. Robustness: the stabilizer

This is not a selection step: $100k was fixed beforehand. It checks whether the conclusion depends on that
arbitrary constant. $1 is close to the raw log ratio (production `fundraising_advantage` uses log1p); $1M damps small
campaigns much more.

In [16]:
robust = {}
for stabilizer in [1.0, 100_000.0, 1_000_000.0]:
    fund_k = sf.fundraising_table(stabilizer=stabilizer)
    rows_k, cal_k = sf.backtest(EVAL_YEARS, data=data, polls=polls, ratings=ratings, fund=fund_k,
                                config=STAGE1_CONFIG, caps=CAPS, weights=WEIGHTS_4)
    table = stage2.score_table(rows_k, data, PREDICTIONS)
    for label in PREDICTIONS:
        robust[(f"${stabilizer:,.0f}", label)] = {
            **table.loc[label, ["mae", "bias", "ex_ante_mae", "accuracy_pct"]],
            "unpolled_mae": (rows_k.loc[~rows_k["polling_available"], "actual"]
                             - rows_k.loc[~rows_k["polling_available"], PREDICTIONS[label]]).abs().mean(),
            "B_slope_2024": cal_k.loc[2024, "B_slope"],
            "at_cap_pct": 100 * rows_k.loc[rows_k["fund_adv"].notna(), "at_cap"].mean()}
pd.DataFrame(robust).T.round(2)

mae  bias  ex_ante_mae  accuracy_pct  unpolled_mae  B_slope_2024  at_cap_pct
$1         baseline (production)  4.76 -1.86         3.58         93.37          5.54          0.68       68.35
           A: 4th component       4.75 -1.99         3.57         93.37          5.55          0.68       68.35
           B: capped adjustment   4.76 -2.00         3.58         94.58          5.47          0.68       68.35
$100,000   baseline (production)  4.76 -1.86         3.58         93.37          5.54          1.15       86.71
           A: 4th component       4.71 -1.98         3.54         93.98          5.46          1.15       86.71
           B: capped adjustment   4.77 -2.00         3.61         94.58          5.48          1.15       86.71
$1,000,000 baseline (production)  4.76 -1.86         3.58         93.37          5.54          1.57       90.51
           A: 4th component       4.74 -1.98         3.53         94.58          5.54          1.57       90.51
           B: capped adjustment   4.77 -2.00         3.61         94.58          5.48          1.57       90.51

## 9. 2026: what each variant would do (descriptive; nothing is adopted)

Fits use 2006/2012–2024. NE (Osborn, independent), ID and SD have no D nominee and get no fundraising signal.

In [17]:
f26, cal26 = sf.forecast(data=data, polls=polls, ratings=ratings, fund=fund, config=STAGE1_CONFIG, caps=CAPS,
                         weights=WEIGHTS_4)
print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in cal26.items()})
COLS = ["fund_status", "fund_adv", "weight_case", "prediction", "margin_fundraising", "prediction_A", "delta_F",
        "prediction_B"]
f26 = f26.set_index(["state", "class"])
display(Markdown("**Within 10 points (production blend)**"))
display(f26.loc[f26["prediction"].abs() <= 10, COLS].sort_values("prediction").round(2))
changed = f26[(np.sign(f26["prediction_A"]) != np.sign(f26["prediction"]))
              | (np.sign(f26["prediction_B"]) != np.sign(f26["prediction"]))]
print("2026 races whose called winner changes:", changed.index.tolist() or "none")
print("D-positive races:", {"baseline": int((f26["prediction"] > 0).sum()), "A": int((f26["prediction_A"] > 0).sum()),
                            "B": int((f26["prediction_B"] > 0).sum())}, "of", len(f26))

{'year': 2026, 'B_slope': np.float64(0.98), 'B_slope_se': np.float64(0.22), 'B_n': 212, 'B_years': '2012-2024', 'S1_slope': np.float64(1.74), 'S1_slope_se': np.float64(0.3), 'A_intercept': np.float64(-3.66), 'A_slope': np.float64(10.36), 'A_n': 268, 'A_years': '2006-2024'}


**Within 10 points (production blend)**

,,fund_status,fund_adv,weight_case,prediction,margin_fundraising,prediction_A,delta_F,prediction_B
state,class,,,,,,,,
SC,2,ok,2.49,no_polling,-9.24,22.13,-7.65,1.00,-8.24
FL,3,ok,-0.25,polling,-5.46,-6.21,-5.56,-0.24,-5.71
IA,2,ok,0.54,polling,-1.10,1.91,-1.01,0.50,-0.60
AK,2,ok,1.16,polling,-0.63,8.35,-0.34,0.50,-0.13
KS,2,ok,1.26,polling,-0.53,9.43,-0.16,0.50,-0.03
TX,2,ok,1.88,polling,1.63,15.79,2.00,0.50,2.13
OH,3,ok,1.23,polling,1.67,9.11,1.88,0.50,2.17
MI,2,ok,0.85,polling,6.98,5.12,6.91,0.50,7.48
ME,2,ok,0.13,polling,8.01,-2.29,7.69,0.13,8.14


2026 races whose called winner changes: none
D-positive races: {'baseline': 17, 'A': 17, 'B': 17} of 35


## Conclusion

**Neither approach adds a reliable signal beyond fundamentals, polls and ratings. Don't adopt either.** Pooled
2016–2024, 166 races (158 with pre-election fundraising; 8 without, all `post_election_only`):

| | Baseline (production) | A: 4th component | B: capped |
|---|---|---|---|
| MAE, all | 4.76 | 4.71 | 4.77 |
| MAE, ex-ante competitive (51) | 3.58 | 3.54 | 3.61 |
| Bias (actual − pred.) | −1.86 | −1.98 | −2.00 |
| Winners called | 93.4% | 94.0% | 94.6% |
| MAE, polled (109) | 4.35 | 4.32 | 4.40 |
| MAE, unpolled (57) | 5.54 | 5.46 | 5.48 |
| Monte Carlo Brier, 2018–24 (S-M0) | 0.0463 | 0.0457 | 0.0451 |

- **A: a small gain, within noise.** MAE improves by 0.05, about one year-clustered SE (0.05), and in only 3 of 5
  years. Races move 0.2 points on average with polls and 0.6 without.
- **B: no MAE gain, and polled races get worse.** It is worse with polls (+0.05; 58 races worse, 45 better) and
  slightly better without (−0.06 ± 0.14). Its winner and Brier gains come from two races: MT 2018 and NV 2022, polled
  Democratic incumbents who outraised and won narrowly.
- **The caps bind almost everywhere.** 87% of races with fundraising reach the cap (84% polled, 91% unpolled), so B
  is in effect ±cap × sign(who raised more), not a graded signal. The diagnostic without caps is clearly worse (MAE
  5.09): the residual slope over-adjusts out of sample, and the caps are what keep B neutral.
- **Both push the bias the wrong way.** The blend is already too Democratic (−1.86), and Democrats outraised in most
  years. In polled races with a Democratic incumbent, B adds +0.4 where the blend already overstates Democrats by
  1.2.
- **The regression-level gain doesn't survive the blend.** In Stage 1, fundraising improves the ex-ante competitive
  MAE (4.98 → 4.55) but worsens the overall MAE (6.01 → 6.49), as in 04. After the blend, the competitive gain is
  0.04 (A) or −0.03 (B): polls and ratings already carry it.
- **The signal is fading.** B's slope falls every window (1.99 for 2016 → 1.15 for 2024 → 0.98 for 2026), so it
  leans on 2012–2016. That includes the 2012–2014 blends, which have no ratings.
- **Robust to the stabilizer** ($1, $100k, $1M): the same picture each time.
- **2026:** neither variant changes a called winner, and the D-positive count stays 17 of 35. B moves AK, KS, IA,
  TX and OH half a point toward Democrats.

**Limitations.**
- Unpolled competitive races, the case this was meant to help most, barely exist in the sample: 3 ex-ante
  competitive unpolled races in 2016–2024. The question can't be settled with this backtest.
- Pre-election coverage is good in the eval years (158 of 166) but thin early: 7 of 31 races in 2006, and 24–27 a
  year in 2008–2014. That thins A's early calibration.
- The 12G amounts are as they stand in today's FEC data, so amendments filed after an election could be included.
  This is minor.
- Five eval years: the year-clustered SEs rest on five units.